# 1.2 · 架构：宿主、客户端与服务器

**Architecture: hosts, clients and servers**

> **学完本课你能：** 凭记忆画出 MCP 的宿主–客户端–服务器架构及其两层结构，追踪一次完整的 discover → list → call → notify 交互，并解释该架构如何让服务器彼此隔离。

官方阅读材料：[Architecture overview](https://modelcontextprotocol.io/docs/2026-07-28/learn/architecture) · [Spec: Architecture](https://modelcontextprotocol.io/specification/2026-07-28/architecture/index)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 两个聚焦的服务器
规范希望服务器<em>极其容易构建</em>并且<em>职责聚焦</em>。这里有两个：一个项目文件服务器和一个工单跟踪器。每个服务器都在自己的审计日志（Audit Trail）中准确记录它收到了什么——这样我们之后就能证明每个服务器能看到什么。

In [ ]:
import json
from contextlib import AsyncExitStack

from mcp import Client
from mcp.server import MCPServer
from mcp.server.mcpserver import Context


def show(obj):
    print(json.dumps(obj, indent=2, default=str))


# --- server 1: project files -------------------------------------------------
files_server = MCPServer("files", version="1.0.0")
FILES = {"notes.txt": "Ship v2 on Friday.", "secret.env": "API_KEY=sk-live-123"}
files_audit = []


@files_server.tool()
def read_file(path: str, ctx: Context) -> str:
    """Read a file from the project folder."""
    files_audit.append({"request": ctx.request_id, "arguments": {"path": path}})
    return FILES[path]


# --- server 2: ticket tracker ------------------------------------------------
tickets_server = MCPServer("tickets", version="0.3.0")
tickets_audit = []


@tickets_server.tool()
def create_ticket(title: str, ctx: Context) -> str:
    """Create a ticket in the issue tracker and return its id."""
    caps = ctx.client_capabilities
    tickets_audit.append({"request": ctx.request_id, "arguments": {"title": title},
                          "protocolVersion": ctx.protocol_version,
                          "clientCapabilities": caps.model_dump(by_alias=True, exclude_none=True) if caps else None})
    return f"T-{len(tickets_audit)}"


print("servers:", files_server.name, "and", tickets_server.name)

### 2. 宿主为每个服务器创建一个客户端
我们的 `Host` 拥有规范赋予宿主（Host）的一切：客户端实例、合并后的工具注册表、对话。每个 `Client` 在连接时发现自己的服务器，因此之后 `protocol_version`、`server_info` 和 `server_capabilities` 都已就绪。

> <strong>SDK 说明。</strong> 用 `AsyncExitStack` 来进入这些客户端：它们必须按打开的相反顺序关闭（anyio 取消作用域）。以其他任何顺序对它们调用 `__aexit__` 都会抛出 `RuntimeError`。

In [ ]:
class Host:
    """A miniature MCP host: one client per server, one tool registry, one conversation."""

    def __init__(self, servers):
        self.servers = servers
        self.clients = {}          # server name -> Client
        self.registry = {}         # tool name exposed to the model -> (server name, real tool name)
        self.conversation = []     # stays in the host; servers never see it

    async def __aenter__(self):
        self._stack = AsyncExitStack()
        for server in self.servers:
            client = await self._stack.enter_async_context(Client(server))
            self.clients[client.server_info.name] = client
        await self.refresh_tools()
        return self

    async def __aexit__(self, *exc):
        await self._stack.aclose()

    async def refresh_tools(self):
        self.registry.clear()
        for server_name, client in self.clients.items():
            for tool in (await client.list_tools()).tools:
                exposed = tool.name
                if exposed in self.registry:                    # two servers, same tool name
                    exposed = f"{server_name}__{tool.name}"     # namespace it for the model
                self.registry[exposed] = (server_name, tool.name)

    async def call(self, exposed_name, arguments):
        server_name, tool_name = self.registry[exposed_name]  # route by name
        result = await self.clients[server_name].call_tool(tool_name, arguments)
        text = result.content[0].text
        self.conversation.append({"tool": exposed_name, "server": server_name, "result": text})
        return text


async with Host([files_server, tickets_server]) as host:
    for name, client in host.clients.items():
        print(f"client -> {name} v{client.server_info.version}  protocol={client.protocol_version}")
        print("   server capabilities:", client.server_capabilities.model_dump(by_alias=True, exclude_none=True))
    print("\nunified tool registry the model would see:")
    show(host.registry)

### 3. 按名称路由工具调用
这就是把文档中的 `handle_tool_call` 伪代码变成真实代码：“模型”指名一个工具（Tool），宿主找到拥有它的客户端，调用它，并把结果追加到对话中。

In [ ]:
files_audit.clear(); tickets_audit.clear()

async with Host([files_server, tickets_server]) as host:
    # pretend the model asked for these two calls, in order
    note = await host.call("read_file", {"path": "notes.txt"})
    secret = await host.call("read_file", {"path": "secret.env"})
    ticket = await host.call("create_ticket", {"title": f"Follow up: {note}"})
    print("host conversation (lives only in the host):")
    show(host.conversation)

### 4. 隔离：每个服务器实际看到了什么？
把宿主的对话与每个服务器的审计日志对比一下。工单服务器只看到了一个标题——从未看到文件服务器返回的密钥，也从未看到对话。数据之所以从一个服务器流到另一个服务器，<strong>仅仅</strong>是因为宿主（Host）选择了把 `note` 放进工单标题。这就是服务器隔离（Server Isolation）。

In [ ]:
print("files server saw:")
show(files_audit)
print("\ntickets server saw:")
show(tickets_audit)

leaked = any("sk-live" in json.dumps(entry) for entry in tickets_audit)
print("\ndid the secret reach the tickets server?", leaked)

现在设想模型被提示注入（Prompt Injection），在调用 `create_ticket` 时把密钥放进了标题。协议会如实地把它传过去——边界的可靠程度取决于宿主的策略。一个真实的宿主会在 `Host.call` 中放置一道同意关卡。

### 5. 能力随每个请求一起传递
在 2026-07-28 中，客户端在<em>每个</em>请求中声明自己的能力（Capability）。同一个服务器会从一个能处理征询（Elicitation）的客户端和一个不能处理的客户端那里看到不同的能力——中间没有任何握手。

In [ ]:
async def handle_elicitation(context, params):
    return None   # never invoked here; its presence makes the client advertise `elicitation`


tickets_audit.clear()
async with Client(tickets_server) as plain:
    await plain.call_tool("create_ticket", {"title": "from a plain client"})
async with Client(tickets_server, elicitation_callback=handle_elicitation) as rich:
    await rich.call_tool("create_ticket", {"title": "from an elicitation-capable client"})

for entry in tickets_audit:
    print(f"{entry['arguments']['title']:<38} -> clientCapabilities = {entry['clientCapabilities']}")

### 6. 名称冲突是宿主的问题
工具（Tool）名称只在服务器<em>内部</em>唯一。再加一个同样有 `read_file` 的服务器，宿主就必须在把工具展示给模型之前给它加上命名空间前缀（Namespacing）。

In [ ]:
wiki_server = MCPServer("wiki", version="2.0.0")


@wiki_server.tool()
def read_file(path: str) -> str:
    """Read a page from the team wiki."""
    return f"wiki page {path}"


async with Host([files_server, tickets_server, wiki_server]) as host:
    show(host.registry)
    print(await host.call("wiki__read_file", {"path": "onboarding"}))

### 7. 文档中的示例，在线路上的样子
现在来看架构页面上的完整序列——discover → `tools/list` → `tools/call` → `subscriptions/listen` → `list_changed` → 刷新——针对一个真实的 stdio 子进程，客户端一侧不使用 SDK。`enable_forecast` 工具会在运行时添加一个新工具，这样服务器就有理由发送通知（Notification）。

In [ ]:
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

workdir = Path(tempfile.mkdtemp())
(workdir / "weather_server.py").write_text(textwrap.dedent('''
    from mcp.server import MCPServer
    from mcp.server.mcpserver import Context

    server = MCPServer("weather", version="1.0.0")

    @server.tool()
    def weather_current(location: str, units: str = "metric") -> str:
        """Get current weather information for a location."""
        return f"{location}: 20 degrees ({units}), partly cloudy"

    @server.tool()
    async def enable_forecast(ctx: Context) -> str:
        """Unlock the multi-day forecast tool."""
        def weather_forecast(location: str, days: int = 3) -> str:
            """Get a multi-day forecast."""
            return f"{location}: sunny for {days} days"
        server.add_tool(weather_forecast)
        await ctx.notify_tools_changed()
        return "forecast enabled"

    if __name__ == "__main__":
        server.run(transport="stdio")
'''))

proc = subprocess.Popen([sys.executable, str(workdir / "weather_server.py")],
                        stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)
META = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientInfo": {"name": "example-client", "version": "1.0.0"},
        "io.modelcontextprotocol/clientCapabilities": {}}


def send(message):
    print(">>>", json.dumps(message)[:220])
    proc.stdin.write(json.dumps(message) + "\n")
    proc.stdin.flush()


def receive_until(request_id):
    """Read lines until the response to `request_id`; notifications may arrive first."""
    while True:
        message = json.loads(proc.stdout.readline())
        kind = "notification" if "id" not in message else "response"
        print(f"<<< ({kind})", json.dumps(message)[:300])
        if message.get("id") == request_id:
            return message


print("--- step 1: discovery")
send({"jsonrpc": "2.0", "id": 1, "method": "server/discover", "params": {"_meta": META}})
discovery = receive_until(1)
print("\nfull discovery result:")
show(discovery["result"])

In [ ]:
print("--- step 2: tool discovery")
send({"jsonrpc": "2.0", "id": 2, "method": "tools/list", "params": {"_meta": META}})
listing = receive_until(2)
print("tools:", [t["name"] for t in listing["result"]["tools"]])

print("\n--- step 3: tool execution")
send({"jsonrpc": "2.0", "id": 3, "method": "tools/call",
      "params": {"name": "weather_current", "arguments": {"location": "San Francisco", "units": "imperial"},
                 "_meta": META}})
call_reply = receive_until(3)

In [ ]:
print("--- step 4: opt in to change notifications (the request id becomes the subscriptionId)")
send({"jsonrpc": "2.0", "id": 4, "method": "subscriptions/listen",
      "params": {"_meta": META, "notifications": {"toolsListChanged": True}}})
print("<<< (notification)", proc.stdout.readline().strip())   # the acknowledgement comes first

print("\n--- something changes the server's tools")
send({"jsonrpc": "2.0", "id": 5, "method": "tools/call",
      "params": {"name": "enable_forecast", "arguments": {}, "_meta": META}})
enable_reply = receive_until(5)

print("\n--- the client reacts by refreshing the list")
send({"jsonrpc": "2.0", "id": 6, "method": "tools/list", "params": {"_meta": META}})
print("tools now:", [t["name"] for t in receive_until(6)["result"]["tools"]])

proc.stdin.close()
print("server exit code:", proc.wait(timeout=10))

看一下顺序：`notifications/tools/list_changed`（没有 `id`，带有 `subscriptionId: 4` 标记）在请求 5 的响应<em>之前</em>就到达了。通知（Notification）是独立的消息；客户端必须随时准备好在订阅流上接收它们。

### 动手练习
1. 在第 3 节中，为 `Host.call` 添加一道<strong>同意关卡</strong>：拒绝（并在对话中记录这次拒绝）任何对 `tickets` 服务器上工具的调用，只要其参数中包含 `"API_KEY"`。用一个试图把密钥作为工单标题提交的调用，重新运行第 3–4 节。
2. 在第 7 节中，发送一个请求 `{"toolsListChanged": True, "promptsListChanged": True}` 的 `subscriptions/listen` 请求，并阅读确认消息。服务器同意响应过滤器的哪一部分？客户端如何得知这一点？
3. 在第 7 节中，向同一个进程分别发送一个 `_meta` 中声明了 `"io.modelcontextprotocol/clientCapabilities": {"elicitation": {}}` 的 `tools/call`，以及一个没有声明的。解释为什么这在 2026-07-28 中是合法的，而在 2025-11-25 的会话中却毫无意义。
4. 在一个 markdown 单元格中勾勒：如果 `files_server` 通过可流式 HTTP（Streamable HTTP）远程运行，你的 `Host` 会如何变化？类中的哪些部分会改变，哪些部分完全不变？为什么？